In [ ]:
import pandas as pd
import chemsource
import os
import sys
import time
import asyncio


sys.path.append(os.path.abspath("../src"))
from harmonization import (
    harmonize_automated_classification,
    harmonize_manual_classification,
)


In [2]:
classified_drug_library_data_path = "../data/drug_library/validation_data_classified_all_3_methods.csv"

harmonized_automated = harmonize_automated_classification(classified_drug_library_data_path)
harmonized_manual = harmonize_manual_classification(classified_drug_library_data_path)


harmonized_automated_non_medical = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'MEDICAL' not in x and "INFO" not in x)]
harmonized_automated_medical = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'MEDICAL' in x and "INFO" not in x)]
harmonized_automated_info = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'INFO' in x)]


In [4]:
drug_library_text = pd.read_csv(classified_drug_library_data_path)
drug_library_text["FEATURE_ID"] = drug_library_text.index
drug_library_text = drug_library_text[["FEATURE_ID", 
    "name_used", 
    "text"]].rename(columns={
    "name_used": "NAME",
    "text": "TEXT"})

In [5]:
non_medical_random_sample_idx = harmonized_automated_non_medical.sample(n=50, 
    random_state=42).index.tolist()
medical_random_sample_idx = harmonized_automated_medical.sample(n=50, 
    random_state=42).index.tolist()
info_random_sample_idx = harmonized_automated_info.sample(n=50, 
    random_state=42).index.tolist()

non_medical_random_sample = drug_library_text.loc[non_medical_random_sample_idx]
medical_random_sample = drug_library_text.loc[medical_random_sample_idx]
info_random_sample = drug_library_text.loc[info_random_sample_idx]



In [6]:
# Configure model

openai_api_key = open("../secrets/openai_api_key.txt").read().strip()
ncbi_api_key = open("../secrets/ncbi_api_key.txt").read().strip()

chem = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

prompt = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


chem.prompt = prompt

data_out_path_non_medical = "../data/random_sample_reruns/non_medical_random_sample_reruns_NEW_RETRY.csv"
data_out_path_medical = "../data/random_sample_reruns/medical_random_sample_reruns.csv"


In [8]:
def classification_to_bits(classification):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    bits = ["1" if category in classification else "0" for category in categories]

    if len(set(classification) - set(categories)) > 0:
        unknown_categories = ",".join(set(classification) - set(categories))
        return "".join(bits)+f"_UNKNOWN({unknown_categories})"
    return "".join(bits)



async def async_run(data_list, model_instance):
    # Get the current running event loop instead of get_event_loop()
    loop = asyncio.get_running_loop()
    # Create tasks for each row - now passing tuples of (name, text)
    futures = [loop.run_in_executor(None, model_instance.classify, name, text) for name, text in data_list]
    # Gather all results
    result = await asyncio.gather(*futures, return_exceptions=True)
    return result

# Test the async function
# In Jupyter, use await directly instead of asyncio.run()
# Now passing tuples of (name, text)
test_data = [("aspirin", "pain reliever"), ("glucose", "sugar molecule")]


In [ ]:
num_repeats = 100
with open(data_out_path_non_medical, "a") as f:
    f.write(",".join(non_medical_random_sample.index.astype(str)) + "\n")

non_medical_random_sample_data = list(non_medical_random_sample[["NAME", "TEXT"]].itertuples(index=False, name=None))

for i in range(num_repeats):

    classifications = await async_run(non_medical_random_sample_data, chem)
    time.sleep(1)
    classifications_bits = [classification_to_bits(c) for c in classifications]
    with open(data_out_path_non_medical, "a") as f:
        f.write(",".join(classifications_bits) + "\n")


In [7]:
non_medical_random_sample_data = pd.read_csv(data_out_path_non_medical, dtype=str)
non_medical_random_sample_data_unique = non_medical_random_sample_data.apply(lambda col: col.value_counts().to_dict())
non_medical_random_sample_data_unique

2352                             {'010000': 100}
1700                             {'001000': 100}
1825                             {'010000': 100}
2498                             {'011100': 100}
1442                             {'001000': 100}
3482                             {'010000': 100}
56                               {'010000': 100}
4453                             {'010000': 100}
4908                {'000010': 57, '000001': 43}
1297                             {'010000': 100}
1551                             {'000010': 100}
4944                             {'001000': 100}
1757                             {'001110': 100}
1637                             {'010100': 100}
2424                             {'011100': 100}
1703                             {'011100': 100}
4926                             {'001000': 100}
3099                {'011000': 88, '010000': 12}
2745                             {'000010': 100}
3314                             {'010000': 100}
163                 

In [9]:
medical_random_sample_text_lengths = medical_random_sample["TEXT"].apply(len)
non_medical_random_sample_text_lengths = non_medical_random_sample["TEXT"].apply(len)

In [18]:
medical_random_sample_text_lengths.sum()

np.int64(300003)

In [20]:
non_medical_random_sample_text_lengths.sum()

np.int64(274661)

In [ ]:
num_repeats = 100
with open(data_out_path_medical, "a") as f:
    f.write(",".join(medical_random_sample.index.astype(str)) + "\n")

medical_random_sample_data = list(medical_random_sample[["NAME", "TEXT"]].itertuples(index=False, name=None))

for i in range(num_repeats):
    
    classifications = await async_run(medical_random_sample_data, chem)
    time.sleep(1)
    classifications_bits = [classification_to_bits(c) for c in classifications]
    with open(data_out_path_medical, "a") as f:
        f.write(",".join(classifications_bits) + "\n")

In [10]:
medical_random_sample_data = pd.read_csv(data_out_path_medical, dtype=str)
medical_random_sample_data_unique = medical_random_sample_data.apply(lambda col: col.value_counts().to_dict())
medical_random_sample_data_unique

2984                                      {'100000': 163}
2682                                      {'100000': 163}
3085                                      {'100000': 163}
4028                                      {'100000': 163}
4116                                      {'110000': 163}
3116                                      {'100000': 163}
3578                                      {'000001': 163}
4341                                      {'100000': 163}
981                                       {'100000': 163}
2884                                      {'100000': 163}
1935                                      {'100000': 163}
4626                                      {'100000': 163}
972                                       {'100000': 163}
2774                                      {'100000': 163}
1070                                      {'100000': 163}
1410    {'010000': 137, '110000': 19, '010100': 6, '11...
1044                                      {'100000': 163}
1265          

In [9]:
reasoning_prompt = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer. The formatting is particularly important, \
so pay attention to this section. Provide the output as an explanation of your \
reasoning followed by the separator term \"EXPLANATION_COMPLETE\" and then \
provide a plain text separated by commas, and in that list, provide only the \
categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO). Provided Information:\n"


chem_reasoning = chemsource.ChemSource(
    prompt=reasoning_prompt,
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    explanation=True,
    explanation_separator="EXPLANATION_COMPLETE",
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", 
                        "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

reasoning_data_out_path_non_medical = "../data/random_sample_reruns/reasoning_non_medical_random_sample_reruns.csv"
reasoning_data_out_path_medical = "../data/random_sample_reruns/reasoning_medical_random_sample_reruns.csv"


In [ ]:
num_repeats = 100
with open(reasoning_data_out_path_non_medical, "a") as f:
    f.write(",".join(non_medical_random_sample.index.astype(str)) + "\n")

non_medical_random_sample_data = list(non_medical_random_sample[["NAME", "TEXT"]].itertuples(index=False, name=None))

for i in range(num_repeats):
    
    classifications = await async_run(non_medical_random_sample_data, chem_reasoning)
    time.sleep(1)
    classifications_bits = [classification_to_bits(c) for c in classifications]
    with open(reasoning_data_out_path_non_medical, "a") as f:
        f.write(",".join(classifications_bits) + "\n")

TypeError: argument of type 'RateLimitError' is not iterable

In [11]:
reasoning_non_medical_random_sample_data = pd.read_csv(reasoning_data_out_path_non_medical, dtype=str)
reasoning_non_medical_random_sample_data_unique = reasoning_non_medical_random_sample_data.apply(lambda col: col.value_counts().to_dict())
reasoning_non_medical_random_sample_data_unique = pd.DataFrame(reasoning_non_medical_random_sample_data_unique)
reasoning_non_medical_random_sample_data_unique["first_val"] = reasoning_non_medical_random_sample_data_unique[0].apply(lambda x: list(x.values())[0])
reasoning_non_medical_random_sample_data_unique.sort_values(by="first_val", ascending=False)
reasoning_non_medical_inconsistent_indices = reasoning_non_medical_random_sample_data_unique[reasoning_non_medical_random_sample_data_unique["first_val"] != 67].index.tolist()

In [12]:
non_medical_random_sample_inconsistent_subset = non_medical_random_sample.loc[[int(item) for item in reasoning_non_medical_inconsistent_indices]]

chem_reasoning_explanations = chemsource.ChemSource(
    prompt=reasoning_prompt,
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    explanation=True,
    output_explanation=True,
    explanation_separator="EXPLANATION_COMPLETE",
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", 
                        "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

reasoning_explanations_data_out_path_non_medical = "../data/random_sample_reruns/reasoning_non_medical_random_sample_reruns_with_explanations.csv"
reasoning_explanations_explanations_out_path_non_medical = "../data/random_sample_reruns/reasoning_non_medical_random_sample_reruns_model_explanations.csv"


In [63]:
non_medical_random_sample_inconsistent_subset

,FEATURE_ID,NAME,TEXT
1700,1700,Sulfanitran,Sulfanitran is a sulfonamide antibiotic which ...
4908,4908,Sodium taurolithocholate,The proportion of sodium taurolithocholate (N...
1757,1757,Formylformic acid,Glyoxylic acid or oxoacetic acid is an organic...
1637,1637,Pantothenic acid,Pantothenic acid (vitamin B5) is a B vitamin a...
1703,1703,Isopentanoic acid,"Isovaleric acid, also known as 3-methylbutanoi..."
4926,4926,Lipid crimson,Sudan IV (C24H20N4O) is a lysochrome (fat-solu...
3099,3099,Isomalt,"Isomalt is a sugar substitute, a mixture of th..."
3314,3314,Trans-vaccenic acid,Vaccenic acid is a naturally occurring trans f...
163,163,Capstar,Nitenpyram is a chemical frequently used as an...
4754,4754,Curdlan,"Curdlan is a water-insoluble linear beta-1,3-g..."


In [13]:
num_repeats = 10
with open(reasoning_explanations_data_out_path_non_medical, "a") as f:
    f.write(",".join(non_medical_random_sample_inconsistent_subset.index.astype(str)) + "\n")

with open(reasoning_explanations_explanations_out_path_non_medical, "a") as f:
    f.write(",".join(non_medical_random_sample_inconsistent_subset.index.astype(str)) + "\n")

non_medical_random_sample_inconsistent_subset = list(non_medical_random_sample_inconsistent_subset[["NAME", "TEXT"]].itertuples(index=False, name=None))

for i in range(num_repeats):
    
    classifications_with_explanations = await async_run(non_medical_random_sample_inconsistent_subset, chem_reasoning_explanations)
    classifications = []
    explanations = []

    for item in classifications_with_explanations:
        curr_classification, curr_explanation = item
        classifications.append(curr_classification)
        explanations.append("\"" + curr_explanation.replace("\"", "") + "\"")
    
    classifications_bits = [classification_to_bits(c) for c in classifications]

     
    with open(reasoning_explanations_data_out_path_non_medical, "a") as f:
        f.write(",".join(classifications_bits) + "\n")
    with open(reasoning_explanations_explanations_out_path_non_medical, "a") as f:
        f.write(",".join(explanations) + "\n")
    
    time.sleep(1)